# NeuroGolf — Validated Task-Level Blend (7184.59)

Publishes a 400-task ONNX archive that scored **7184.59** on the public leaderboard.

## Lineage & acknowledgements

This archive is a task-level blend built on top of public work — thank you to the authors:

- `kokinnwakashuu/neurogolf-graph-golf-continuation` (7184.38 base archive; Ryosuke 7184+ bundle + Akihiro task replacements)
- `franksunp/neurogolf-consolidated-audit` v93 (task219 rewire, task233, task338)
- `boristown/neurogolf-stack-audit-variant` line (earlier 7178.62 / 7180.86 bases)

On top of the public frontier, this blend keeps **original validated graph optimizations** on 12 tasks
(exact-output tensor merges + common-subexpression elimination), each verified against
train + test + full ARC-GEN and confirmed on the leaderboard:
`task018, task023, task029, task090, task096, task118, task133, task157, task209, task233, task319, task338, task397`.

## Integrity

The hidden payload below is checked for byte size, SHA-256, and all 400 canonical task names
before writing `/kaggle/working/submission.zip`. Nothing is rebuilt at run time.


In [ ]:
import json
RECEIPT = '{\n  "public_score": 7184.59,\n  "submission_bytes": 519345,\n  "submission_sha256": "d9c4360a5041eb554ce5ee93edea668bdc477d6169ac8f06e1e32b6c93ecf98c",\n  "task_count": 400,\n  "base_archives": [\n    {\n      "ref": "kokinnwakashuu/neurogolf-graph-golf-continuation",\n      "public_score": 7184.38\n    },\n    {\n      "ref": "franksunp/neurogolf-consolidated-audit",\n      "version": 93,\n      "public_score": 7184.44\n    }\n  ],\n  "original_contributions": {\n    "method": "exact-output tensor merge (validated on train+test+full arc-gen) + CSE",\n    "tasks": [\n      "task018",\n      "task023",\n      "task029",\n      "task090",\n      "task096",\n      "task118",\n      "task133",\n      "task157",\n      "task209",\n      "task233",\n      "task319",\n      "task338",\n      "task397"\n    ],\n    "excluded_after_probe": [\n      "task101",\n      "task366"\n    ]\n  }\n}'
print(json.loads(RECEIPT) and RECEIPT)

In [ ]:
import base64
import hashlib
import re
import zipfile
from pathlib import Path

OUT = Path('/kaggle/working/submission.zip')
EXPECTED_BYTES = 519345
EXPECTED_SHA256 = 'd9c4360a5041eb554ce5ee93edea668bdc477d6169ac8f06e1e32b6c93ecf98c'
TASK_RE = re.compile(r'^task(\d{3})\.onnx$')

B64 = ''.join([
    '...',
    'WgAA+5EHAAAA'
])

blob = base64.b64decode(B64)
assert len(blob) == EXPECTED_BYTES, f'byte mismatch: {len(blob)}'
assert hashlib.sha256(blob).hexdigest() == EXPECTED_SHA256, 'sha256 mismatch'
OUT.write_bytes(blob)

with zipfile.ZipFile(OUT) as zf:
    names = sorted(zf.namelist())
    assert len(names) == 400, f'expected 400 tasks, got {len(names)}'
    nums = sorted(int(TASK_RE.match(n).group(1)) for n in names if TASK_RE.match(n))
    assert nums == list(range(1, 401)), 'task names not canonical'
print('submission.zip written:', OUT.stat().st_size, 'bytes — integrity OK')
